In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import sys
import json
from datetime import datetime

In [ ]:
from typing import get_origin, get_args, Literal

In [ ]:
load_dotenv()

OPEN_ROUTER_API = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPEN_ROUTER_API_KEY")
MODEL = os.getenv("OPENROUTER_MODEL") or os.getenv("MODEL")

if not OPEN_ROUTER_API:
    raise ValueError("API key not found. Please set OPENROUTER_API_KEY in .env file.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPEN_ROUTER_API,
)


In [ ]:
class Memory:
    def __init__(self):
        self.data = []
        self.next_id = 1

    def remember(self, key, value):
        for memory in self.data:

            if memory["key"] == key:
                memory["value"] = value
                memory["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                return

        self.data.append({
            "id" : self.next_id,
            "key": key,
            "value": value,
            "timestamp" : datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        
        self.next_id += 1
    
    def recall(self, key):
        for memory in reversed(self.data):
            if memory["key"] == key:
                return memory["value"]

    def forget(self, key):
        self.data = [
            memory
            for memory in self.data
            if memory["key"] != key
        ]

    def list_memories(self):
        return self.data

    def search(self, query):
        results = []

        for memory in self.data:
            if query.lower() in memory["key"].lower():
                results.append(memory)

        return results

In [ ]:
class Tool:
    def __init__(self, function, description):
        self.function = function
        self.description = description
        self.parameters = generate_parameters(function)

    def execute(self, arguments, context):
        try:
            if context is None:
                context = {}
            
            return self.function(**arguments, **context)
        except Exception as e:
            return f"Tool execution failed: {str(e)}"

    def schema(self):
        return {
            "type" : "function",
            "function":{
                "name": self.function.__name__,
                "description": self.description,
                "parameters": self.parameters
            }
        }
    

In [ ]:
class Agent:
    def __init__(self, client, tool_list, model, system_prompt):
        self.client = client
        self.tool_list = tool_list
        self.model = model

        self.TOOL_MAP = {
            tool.function.__name__ : tool
            for tool in tool_list
        }

        self.Tool_SCHEMA = [
            tool.schema()
            for tool in tool_list
        ]

        self.messages = [{
            "role": "system",
            "content": system_prompt
        }       
        ]

        self.state = {
        }

        self.memory = Memory()
        
        self.context = {
            "memory" : self.memory
        }
        
    def set_state(self, key, value):
        self.state[key] = value

    def get_state(self, key):
        return self.state[key]

    def call_llm(self):
        return client.chat.completions.create(
                model=self.model,
                messages=self.messages,
                tools=self.Tool_SCHEMA,
                max_tokens=1000
            )

    def execute(self, tool_call):
            tool_name = tool_call.function.name
            print(f"TOOL CALLED: {tool_name}")

            tool = self.TOOL_MAP.get(tool_name)

            if not tool:
                return f"Tool '{tool_name}' does not exist."

            try:
                arguments = json.loads(tool_call.function.arguments)
                return tool.execute(arguments, self.context)

            except Exception as e:
                return f"Tool execution failed: {str(e)}"
        
    
    def run(self, user_input):
        self.messages.append({"role": "user", "content": user_input})

        MAX_ITERATIONS = 10

        for iteration in range(MAX_ITERATIONS):
            
            response = self.call_llm()

            response_message = response.choices[0].message
            self.messages.append(response_message)

            if not response_message.tool_calls:
                return f"AI: ",response_message.content

            for tool_call in response_message.tool_calls:
                    print("Tool is running")
                    result = self.execute(tool_call)
                    
                    self.messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_call.function.name,
                        "content": json.dumps(result)
                    })

        
        else:
            print("MAX Tool iterations reached!!")
            
        

In [ ]:
def python_type_to_json_type(annotation):

    if get_origin(annotation) is Literal:

        values = get_args(annotation)

        first_value = values[0]

        if isinstance(first_value, str):
            json_type = "string"
        
        elif isinstance(first_value, int):
            json_type = "integer"

        elif isinstance(first_value, float):
            json_type = "number"
        
        elif isinstance(first_value, bool):
            json_type = "boolean"

        else:
            json_type = "string"

        return{
            "type": json_type,
            "enum": list(values)
        }
    if annotation == str:
        return "string"

    elif annotation == int:
        return "integer"

    elif annotation == float:
        return "number"

    elif annotation == bool:
        return "boolean"

    return "string"

In [ ]:
import inspect

def generate_parameters(function):
    
    signature = inspect.signature(function)

    properties = {}
    required = []

    for name, parameter in signature.parameters.items():
        if name == "memory":
            continue

        json_type = python_type_to_json_type(parameter.annotation)

        properties[name] = {
            "type" : json_type
        }

        if parameter.default is inspect.Parameter.empty:
            required.append(name)

    return {
        "type" : "object",
        "properties" : properties,
        "required" : required
    }

In [ ]:
def get_current_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
time_tool = Tool(
    function = get_current_time,
    description="Get current Time",
)

In [ ]:
def calculator(a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]):
    if(operation == "add"):
        return a+b

    elif(operation == "divide"):
        if(b != 0):
            return a/b
        else: return "Cannot divide with Zero"

    elif(operation == "subtract"):
        return a-b
    
    elif(operation == "multiply"):
        return a*b


In [ ]:
calculator_tool = Tool(
    function=calculator,
    description="Perform mathematical calculations",
)

In [ ]:
def greet(name: str, age: int, excited: bool = False):
    if excited:
        return f"Hello {name}! You are {age} years old!"
    return f"Hello {name}. You are {age} years old."

In [ ]:
greet_tool = Tool(
    greet,
    "Greets a Person"
)

In [ ]:
def save_memory(memory: Memory, key: str, value: str):
    memory.remember(key, value)
    return f"Remembered {key} = {value}"

In [ ]:
memory_tool = Tool(
    save_memory,
    "Saves important information to agent's memory"
)

In [ ]:
def recall_memory(memory: Memory, key: str):
    value = memory.recall(key)
    if value is None:
        return f"No memory found for '{key}'"

    return f"The value of {key} = {value}"

In [ ]:
recall_tool = Tool(
    recall_memory,
    """Retrieve a memory using its exact key.

    Use this ONLY when you already know the exact memory key.
    For example, if the key is exactly "hometown", use:
    recall_memory(key="hometown").

    If you do not know the exact key, use search_memory instead."""
)

In [ ]:
def forget_memory(memory: Memory, key: str):
    memory.forget(key)
    return f"memory forgotten"

In [ ]:
forget_tool = Tool(
    forget_memory,
    "Used to forget a memory from agent's memory"
)

In [ ]:
def get_all_memories(memory: Memory):
    return memory.list_memories()

In [ ]:
def search(memory: Memory, query: str):
    results =  memory.search(query)

    if not results:
        return f"No memories found for '{query}'"

    return results

In [ ]:
search_memory_tool = Tool(
    search,
    """Search the agent's memory using keywords when you are unsure of the
    exact memory key. Use this tool when the user asks about something
    that may be stored in memory but you do not know the exact key.

    Example:
    User asks "What programming language do I like?"
    Search using query="language".

    Do NOT use recall_memory unless you know the exact key."""
)

In [ ]:
list_memory_tool = Tool(
    get_all_memories,
    "get all the memories currently stored by the agent"
)

In [ ]:
tool_list = [
    calculator_tool,
    time_tool,
    greet_tool,
    memory_tool,
    recall_tool,
    forget_tool,
    search_memory_tool
]

In [ ]:
agent = Agent(
    client=client,
    tool_list=tool_list,
    model = MODEL,
    system_prompt = """You are a helpful AI agent.

You have access to tools for calculations, getting the current time,
and managing memory.

Use the calculator for mathematical calculations.
Use the time tool when the user asks for the current time.

Memory rules:

1. When the user explicitly asks you to remember something,
   use save_memory.

2. If you know the exact memory key, you may use recall_memory.

3. If you do NOT know the exact memory key, use search_memory.
   Do not guess the key.

4. Never use get_all_memories.

5. Do not invent memories."""
)

In [ ]:
while True:
    try:
        user_input = input("You: ")
    except (EOFError, KeyboardInterrupt):
        break

    if not user_input.strip():
        continue

    if user_input.lower().strip() == "exit":
        break

    try:
        response = agent.run(user_input)
        print(response)
    except Exception as e:
        print("Error:", e)


In [ ]:
print(agent.memory.data)